In [ ]:
import os, sys

# Get the absolute path to the project root (one level up from notebooks/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add it to sys.path if not already there
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("sys.path[0]:", sys.path[0])

import os
import yaml
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from src.models.emotion_cnn import build_emotion_model

with open("../config.yaml") as f:
    config = yaml.safe_load(f)

csv_path = config["paths"]["fer_csv"]
model_path = config["paths"]["emotion_model"]


In [ ]:
from src.models.emotion_cnn import build_emotion_model


In [ ]:
IMG_SIZE = 48

fer = pd.read_csv(csv_path)

X = np.array([
    np.fromstring(p, dtype=int, sep=" ") for p in fer["pixels"].tolist()
])
X = X.reshape(-1, IMG_SIZE, IMG_SIZE, 1).astype("float32") / 255.0

y = to_categorical(fer["emotion"].values, num_classes=7)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

X_train.shape, X_val.shape, X_test.shape


In [ ]:
from tensorflow.keras.optimizers import Adam

model = build_emotion_model(input_shape=(48,48,1), num_classes=7)
model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=0.0005),
    metrics=["accuracy"],
)
model.summary()


In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint(model_path, monitor="val_accuracy", save_best_only=True, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=64,
    verbose=1,
    callbacks=callbacks
)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.title("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.title("Loss")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")
